# Estimación colectiva de M&Ms y predicción de respuestas humanas

Se analiza un solo frasco. La unidad de análisis es la respuesta registrada y se asume provisionalmente una respuesta por persona. Se estudian dos preguntas: cómo resumir la percepción colectiva y si conocer la carrera permite anticipar la respuesta de otra persona en este contexto.

Esta copia conserva el código y la discusión. Para ejecutarla, consulte la sección [Reproducción](README.md#reproducción) del README.

## 1. Auditoría y limpieza

El inventario inicial contenía el CSV y cuatro directorios vacíos (`.agents`, `.aws`, `.codex`, `.git`). La columna sin nombre está vacía. `Promedio` muestra un resumen de hoja de cálculo y se verifica contra la media. El proyecto carece de un conteo físico documentado. Se recorta el espacio final en una etiqueta. Se conservan las 14 categorías tal como fueron registradas, incluidas las siglas y las dos etiquetas de profesorado, cuya equivalencia se desconoce. Se preservan coincidencias `(carrera, prediccion)` y se agrega la línea del CSV para trazabilidad. La línea es un identificador técnico, excluido del modelo.

In [ ]:
import hashlib
import json

import joblib
import numpy as np
from IPython.display import Markdown, display

from src.analisis import ORIGINAL, RAIZ, cargar, descriptivos, bootstrap, por_carrera
from src.modelado import evaluar, ajustar_final
from src.figuras import exportar

resultados = RAIZ / "resultados"
procesados = RAIZ / "datos" / "procesados"
modelos = RAIZ / "modelos"
for carpeta in (resultados, procesados, modelos):
    carpeta.mkdir(parents=True, exist_ok=True)

datos, auditoria = cargar()
huella = hashlib.sha256(ORIGINAL.read_bytes()).hexdigest()
auditoria.update(sha256_original=huella, cantidad_real_documentada=False,
                 categorias=int(datos.carrera.nunique()),
                 promedio_auxiliar_coincide=bool(np.isclose(
                     float(auditoria["valores_columna_promedio"][0]), datos.prediccion.mean())))
assert auditoria["promedio_auxiliar_coincide"]
(resultados / "auditoria.json").write_text(
    json.dumps(auditoria, ensure_ascii=False, indent=2), encoding="utf-8")
datos.to_csv(procesados / "estimaciones.csv", index=False)
display(auditoria, datos.dtypes.to_frame("tipo"), datos.carrera.value_counts().to_frame("n"))

## 2. Distribución y estimación colectiva

La media $\bar x=\sum_i x_i/n$ hace participar a todos los valores con igual peso. La mediana es el percentil 50 y minimiza la suma de distancias absolutas. La moda se obtiene por frecuencia. Usamos $s^2=\sum_i(x_i-\bar x)^2/(n-1)$ y $CV=100s/\bar x$. Los cuartiles usan interpolación lineal. La asimetría y el exceso de curtosis cuentan con corrección de sesgo (la normal tiene exceso cero). Las cercas $Q_1-1.5\,IQR$ y $Q_3+1.5\,IQR$ señalan observaciones para revisión, sin eliminarlas del análisis principal. El escenario sin el máximo retira una sola fila.

La media recortada al 10 % promedia las observaciones centrales tras retirar $\lfloor 0.10n\rfloor$ de cada cola. La winsorizada reemplaza esas observaciones por el valor retenido más cercano. Para cada estimador se remuestrean 20 000 conjuntos de $n$ filas con reemplazo y se toman los percentiles 2.5 y 97.5 de su distribución bootstrap. Cada intervalo aproxima la incertidumbre muestral del estadístico correspondiente bajo muestreo independiente y representativo de una población comparable. La representatividad y la independencia se desconocen. Los intervalos pueden tener cobertura imperfecta por redondeos, empates, extremos poco frecuentes o dependencia entre respuestas. Se refieren al centro de las estimaciones humanas.

In [ ]:
resumen, extremos, sensibilidad = descriptivos(datos)
estimadores = bootstrap(datos.prediccion)
for nombre, tabla in {"descriptivos": resumen, "extremos": extremos,
                      "sensibilidad_extremos": sensibilidad, "estimadores": estimadores}.items():
    tabla.to_csv(resultados / f"{nombre}.csv", index=nombre == "descriptivos")
display(resumen.to_frame(), extremos, sensibilidad, estimadores.round(2))

## 3. Comparaciones entre carreras

Todas las categorías se describen. La desviación de un grupo de una persona queda indefinida. Para la aproximación ji-cuadrada de Kruskal–Wallis se consideran solo grupos con cinco o más respuestas. Los grupos pequeños conservan su presencia en el análisis descriptivo y predictivo.

Se examinan el tamaño, la asimetría, Shapiro–Wilk por grupo y Levene centrado en la mediana. Las pruebas de supuestos tienen poca potencia con grupos pequeños. La cola larga, la asimetría y el desequilibrio de tamaños debilitan el ANOVA clásico para estos datos. Kruskal–Wallis compara rangos con corrección por empates y requiere independencia entre observaciones. Una interpretación como diferencia de medianas exigiría distribuciones de forma y dispersión comparables. Se usa como contraste ómnibus exploratorio, sin pruebas por pares.

In [ ]:
grupos, supuestos, contrastes = por_carrera(datos)
grupos.to_csv(resultados / "carreras.csv")
supuestos.to_csv(resultados / "supuestos.csv", index=False)
contrastes.to_csv(resultados / "contrastes.csv", index=False)
display(grupos.round(2), supuestos.round(4), contrastes.round(4))

## 4. Predicción de la estimación humana

La entrada es la carrera, codificada con one-hot dentro de cada ajuste. El objetivo $y$ es `prediccion`. Como el frasco fue el mismo para todos, disponer de su conteo permitiría evaluar el error de los agregados contra esa referencia, pero repetirlo por fila dejaría un objetivo constante. Para aprender a estimar cantidades reales harían falta múltiples frascos contados, características observables y evaluación con frascos separados.

Se usan cinco repeticiones de cinco pliegues externos aleatorios compartidos por todos los métodos. La partición a nivel de fila evalúa nuevas respuestas individuales en condiciones similares, bajo el supuesto de independencia. Las categorías con una fila necesariamente aparecen como desconocidas fuera de muestra y el codificador las transforma en un vector de ceros. El desempeño en categorías ausentes se identifica en las tablas y es exploratorio. Dejar una carrera completa fuera probaría una pregunta distinta.

En cada pliegue externo, cuatro pliegues internos eligen profundidad y tamaño mínimo de hoja mediante MAE. Se fijan 100 árboles por ajuste. Una hoja mayor puede juntar categorías y limitar ajustes a grupos escasos. Con una sola variable categórica, los árboles solo pueden agrupar carreras y producir una predicción constante para cada carrera en cada ajuste. Los modelos base predicen la media o mediana de entrenamiento.

Definimos el residual $e_i=y_i-\hat y_i$, $MAE=\sum_i|e_i|/n$, $RMSE=\sqrt{\sum_i e_i^2/n}$ y $R^2=1-\sum_i e_i^2/\sum_i(y_i-\bar y)^2$. RMSE penaliza más los errores grandes. Un $R^2$ negativo indica error cuadrático por encima de la referencia algebraica de la media del conjunto evaluado. Se calculan las métricas sobre las 92 predicciones externas de cada repetición. La dispersión entre repeticiones refleja sensibilidad a la partición y no constituye un intervalo de confianza independiente. Evaluamos sobreajuste con el MAE de entrenamiento y de prueba sobre los mismos pliegues.

In [ ]:
predicciones, pliegues, repeticiones, grilla = evaluar(datos)
metricas = repeticiones.groupby("modelo")[["MAE", "RMSE", "R2"]].agg(["mean", "std", "min", "max"])
brecha = pliegues.assign(brecha_MAE=lambda t: t.MAE - t.MAE_entrenamiento).groupby(
    "modelo")[["MAE_entrenamiento", "MAE", "brecha_MAE"]].mean()
frecuencias = grilla.assign(profundidad=grilla.profundidad.fillna("Sin límite").astype(str))
elecciones = frecuencias.loc[frecuencias.seleccionado].groupby(
    ["profundidad", "hoja_minima"]).size().rename("selecciones")
bosque = predicciones.loc[predicciones.modelo.eq("Bosque aleatorio")]
residuos = bosque.assign(error_absoluto=bosque.residuo.abs()).groupby("carrera_vista").agg(
    casos_evaluados=("residuo", "size"), residuo_medio=("residuo", "mean"),
    MAE=("error_absoluto", "mean"))
comparacion = repeticiones.pivot(index="repeticion", columns="modelo", values="MAE")
comparacion["diferencia_bosque_menos_dummy_mediana"] = (
    comparacion["Bosque aleatorio"] - comparacion["Dummy mediana"])

for nombre, tabla, indice in [
    ("predicciones_externas", predicciones, False), ("metricas_pliegues", pliegues, False),
    ("metricas_repeticiones", repeticiones, False), ("metricas_resumen", metricas, True),
    ("hiperparametros", grilla, False), ("seleccion_hiperparametros", elecciones, True),
    ("brecha_entrenamiento_prueba", brecha, True), ("residuos_carrera_vista", residuos, True),
    ("comparacion_pareada_mae", comparacion, True),
]:
    tabla.to_csv(resultados / f"{nombre}.csv", index=indice)
display(metricas.round(3), brecha.round(2), elecciones.to_frame(), comparacion.round(2), residuos.round(2))

ajuste = ajustar_final(datos)
joblib.dump(ajuste.best_estimator_, modelos / "bosque_estimaciones_humanas.joblib")
(modelos / "hiperparametros_finales.json").write_text(json.dumps(ajuste.best_params_, indent=2))
display(ajuste.best_params_)

El bosque serializado es un reajuste sobre las 92 filas para inspección. Su rendimiento esperado se estudia exclusivamente mediante las predicciones externas de la validación anidada.

## 5. Figuras

1. El histograma, la densidad suavizada y la caja muestran concentración de respuestas y cola derecha. El suavizado gaussiano es descriptivo y puede distorsionar la densidad cerca del límite inferior.
2. Los puntos por carrera representan respuestas individuales. Los segmentos cubren el intervalo intercuartílico y los rombos marcan medianas. El desplazamiento vertical evita el solapamiento y el eje logarítmico facilita ver valores dispersos.
3. Los puntos representan métricas fuera de muestra por repetición y los rombos indican sus medias.
4. La diagonal indica predicción exacta de la respuesta humana. Los residuos positivos corresponden a subestimaciones de esa respuesta y los negativos a sobreestimaciones.

In [ ]:
exportar(datos, grupos, estimadores, predicciones, repeticiones)

## 6. Discusión científica

La mediana resume la respuesta típica de la muestra con una regla que conserva el mismo peso por participante y resiste mejor la cola superior que la media. Las medias recortada y winsorizada permiten analizar cuánta influencia ejercen los extremos sin presuponer que sean errores. Los tamaños desiguales hacen que el resumen colectivo refleje principalmente a las carreras más representadas. La exactitud de cualquiera de estos agregados requiere el conteo físico.

El contraste por carreras es exploratorio y concierne únicamente a los grupos que alcanzan el umbral muestral. El predictor de carrera tampoco identifica por sí solo mecanismos: las diferencias pueden relacionarse con la composición de la muestra, la forma de reclutamiento, distintas oportunidades de observar el frasco o respuestas comunicadas entre participantes. Estas explicaciones son hipótesis. El diseño y el CSV carecen de metadatos para evaluarlas.

Para un siguiente experimento convendría documentar el conteo mediante verificación independiente, recoger estimaciones individuales antes de que se compartan respuestas y describir reclutamiento y condiciones de observación. Con varios frascos y mediciones asociadas podría formularse un problema predictivo de conteo real. El [informe de resultados](README.md) sintetiza la evidencia cuantitativa y sus implicaciones para el cartel.

En esta muestra, la mediana fue **300 M&Ms** (IC bootstrap del 95 % de 250 a 350). Se mantuvo en 300 en ambos escenarios de sensibilidad. La media pasó de 448,54 a 418,31 al retirar el máximo y a 331,38 al retirar las ocho observaciones por encima de la cerca de Tukey. Este cambio ilustra la influencia de la cola derecha.

Kruskal–Wallis dio $H=7,19$ y $p=0,0662$ en cuatro grupos con 73 respuestas. La evidencia fue insuficiente para distinguir sus distribuciones al nivel del 5 %. Los otros diez grupos quedaron fuera del contraste. Shapiro–Wilk detectó desviaciones de la normalidad en Actuaría y MAC. El resultado de Levene es compatible con varianzas similares, con la incertidumbre propia del tamaño de los grupos.

El bosque obtuvo un MAE de 311,89 M&Ms frente a 298,91 para la media de entrenamiento y 261,37 para la mediana. Perdió frente a la mediana en las cinco repeticiones y también tuvo el peor RMSE y el menor R² en promedio. Su MAE de pliegue aumentó en 31,56 M&Ms de entrenamiento a prueba, compatible con sobreajuste. La elección de profundidad uno en 23 de 25 pliegues muestra que la búsqueda favoreció árboles poco profundos. En la primera repetición, las cuatro estimaciones de 1 500 M&Ms o más dejaron residuos positivos superiores a 1 000 M&Ms. Los 30 casos de carreras ausentes corresponden a solo seis respuestas reutilizadas en cinco repeticiones, por lo que su MAE es especialmente incierto. La evidencia predictiva favorece las referencias simples en este conjunto.

## Referencias

- Galton, F. (1907). Vox Populi. *Nature*, 75, 450–451. [DOI](https://doi.org/10.1038/075450a0)
- Efron, B. (1979). Bootstrap Methods: Another Look at the Jackknife. *The Annals of Statistics*, 7(1), 1–26. [DOI](https://doi.org/10.1214/aos/1176344552)
- Kruskal, W. H. y Wallis, W. A. (1952). Use of Ranks in One-Criterion Variance Analysis. *Journal of the American Statistical Association*, 47(260), 583–621. [DOI](https://doi.org/10.1080/01621459.1952.10483441)
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5–32. [DOI](https://doi.org/10.1023/A:1010933404324)
- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M. y Duchesnay, É. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, 12(85), 2825–2830. [Artículo](https://www.jmlr.org/papers/v12/pedregosa11a.html)
- Proyecto scikit-learn. (s. f.). Documentación de scikit-learn 1.8. [RandomForestRegressor](https://scikit-learn.org/1.8/modules/generated/sklearn.ensemble.RandomForestRegressor.html), [DummyRegressor](https://scikit-learn.org/1.8/modules/generated/sklearn.dummy.DummyRegressor.html) y [métricas de regresión](https://scikit-learn.org/1.8/modules/model_evaluation.html#regression-metrics).
- Proyecto SciPy. (s. f.). *Statistical functions*. [Documentación de scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html).
